# BEIR offline retrieval gate

Internal regression gate: hybrid retrieval (all-MiniLM dense + BM25 sparse,
feature reranker) on BEIR FiQA and SciDocs, capped at 30 queries / 500
documents per dataset, seed 42, 1,000 bootstrap iterations — the exact
parameters the CI `retrieval-gate` workflow uses. This is what protects the
current code against retrieval regressions on every PR.

Data: `evals/notebooks/data/beir_cross_corpus_2026-06-11.json` (current code)
and `beir_regression_2026-06-11.json` (paired-query gate vs. the checked-in
`beir_v1.json` baseline).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import matplotlib.pyplot as plt
import numpy as np
from memd_plotting import (apply_house_style, color_for, load_json,
                            beir_datasets, normalized_summary, figures_dir)
apply_house_style()
FIG = figures_dir()
report = load_json('data/beir_cross_corpus_2026-06-11.json')
datasets = beir_datasets(report)
norm = normalized_summary(report)
regression = load_json('data/beir_regression_2026-06-11.json')
[(d['name'], d['ndcg_at_k'].get('10')) for d in datasets]

## Figure 1 — nDCG@k retrieval curves

nDCG at increasing cutoffs per dataset, plus the macro-averaged
cross-corpus curve. FiQA (financial QA) retrieves cleanly; SciDocs
(citation-style, high semantic drift) is the harder corpus.

In [ ]:
ks = [1, 5, 10, 100]
fig, ax = plt.subplots(figsize=(7.8, 4.8))
for d in datasets:
    ys = [d['ndcg_at_k'].get(str(k)) for k in ks]
    ax.plot(ks, ys, marker='o', linewidth=2, color=color_for(d['name']),
            label=d['name'])
    for k, y in zip(ks, ys):
        if y is not None:
            ax.annotate(f'{y:.3f}', (k, y), textcoords='offset points',
                        xytext=(0, 8), fontsize=8, ha='center', color='#333333')
norm_ndcg = norm.get('ndcg_at_k', {})
if norm_ndcg:
    ys = [norm_ndcg.get(str(k)) for k in ks]
    ax.plot(ks, ys, marker='s', linewidth=2.4, linestyle='--',
            color=color_for('cross-corpus'), label='cross-corpus (macro avg)')
ax.set_xscale('log')
ax.set_xticks(ks)
ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel('cutoff k')
ax.set_ylabel('nDCG@k (higher is better)')
ax.set_ylim(0, 0.75)
ax.set_title('BEIR retrieval: nDCG@k by dataset')
ax.legend(loc='upper left', fontsize=9)
fig.tight_layout()
fig.savefig(FIG / 'beir_ndcg_curves.png')
fig.savefig(FIG / 'beir_ndcg_curves.svg')
plt.show()

## Figure 2 — recall / MRR / precision with 95% CIs

The headline metrics per dataset, with bootstrap 95% confidence intervals.
The wide CIs reflect the 30-query cap — this gate is a fast regression
tripwire, not a precise leaderboard.

In [ ]:
metrics = [('recall', 'Recall@10'), ('mrr', 'MRR'), ('precision', 'P@10')]
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=False)
for ax, (key, title) in zip(axes, metrics):
    names = [d['name'] for d in datasets]
    means = [d[key]['mean'] for d in datasets]
    los = [d[key]['mean'] - d[key]['ci_lower'] for d in datasets]
    his = [d[key]['ci_upper'] - d[key]['mean'] for d in datasets]
    colors = [color_for(n) for n in names]
    bars = ax.bar(range(len(datasets)), means, color=colors, width=0.6,
                  yerr=[los, his], capsize=5, edgecolor='white',
                  error_kw={'ecolor': '#555555', 'lw': 1.2})
    ax.set_xticks(range(len(datasets)))
    ax.set_xticklabels(names, rotation=12, ha='right', fontsize=9)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='x', visible=False)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, 0.02, f'{m:.3f}',
                ha='center', va='bottom', fontsize=9, color='#222222')
fig.suptitle('BEIR metrics with 95% bootstrap CIs')
fig.tight_layout()
fig.savefig(FIG / 'beir_metrics_ci.png')
fig.savefig(FIG / 'beir_metrics_ci.svg')
plt.show()

## Figure 3 — regression gate: baseline vs. current code

The paired-query nDCG@10 gate. The current code (after the v0.60/0.61
quality work) clears the checked-in baseline with a statistically
significant improvement — wins outnumber losses across the paired queries.

In [ ]:
m = regression['metrics'][0]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.5, 4.2),
                               gridspec_kw={'width_ratios': [1, 1.1]})
# left: baseline vs candidate mean nDCG@10
labels = ['baseline\n(beir_v1)', 'current code']
means = [m['baseline_mean'], m['candidate_mean']]
colors = [color_for('baseline'), color_for('candidate')]
bars = axL.bar(labels, means, color=colors, width=0.6, edgecolor='white')
for bar, v in zip(bars, means):
    axL.text(bar.get_x() + bar.get_width()/2, v + 0.008, f'{v:.3f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
axL.set_ylabel('mean nDCG@10')
axL.set_ylim(0, max(means) * 1.25)
axL.set_title('Paired-query nDCG@10')
axL.grid(axis='x', visible=False)
delta = m['candidate_mean'] - m['baseline_mean']
axL.annotate(f'Δ +{delta:.3f}\np = {m["p_value"]:.4f}\neffect {m["effect_size"]:.2f}',
             xy=(1, m['candidate_mean']), xytext=(0.45, max(means) * 1.05),
             fontsize=9, color=color_for('accent'), fontweight='bold')
# right: win / loss / tie breakdown
wlt = [m['wins'], m['losses'], m['ties']]
wlt_labels = [f"wins ({m['wins']})", f"losses ({m['losses']})", f"ties ({m['ties']})"]
wlt_colors = [color_for('memd'), color_for('accent'), '#BBBBBB']
axR.barh(range(3), wlt, color=wlt_colors, edgecolor='white')
axR.set_yticks(range(3))
axR.set_yticklabels(wlt_labels)
axR.invert_yaxis()
axR.set_xlabel(f"paired queries (n = {m['n_pairs']})")
axR.set_title('Per-query outcome vs. baseline')
axR.grid(axis='y', visible=False)
for i, v in enumerate(wlt):
    axR.text(v + 0.4, i, str(v), va='center', fontsize=10)
verdict = 'PASS' if regression.get('overall_passed') else 'FAIL'
fig.suptitle(f'BEIR regression gate — {verdict}', color=color_for('memd'))
fig.tight_layout()
fig.savefig(FIG / 'beir_regression_gate.png')
fig.savefig(FIG / 'beir_regression_gate.svg')
plt.show()